# vLLM 用 AWQ+MTP モデル Dataset 作成

HuggingFace から `twolven/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-AWQ-MTP` をダウンロードし、Kaggle Dataset 化する。

**手順**

1. このノートブックを Kaggle で実行（GPU不要・Internet On）
2. 出力された Dataset URL を控える
3. `build_step2_vllm.py` でその Dataset ID を指定して Step2 notebook を生成

**事前準備**

- Kaggle Secrets に `HF_TOKEN` を登録（gated モデルの場合）
- 必要なら HuggingFace 側で対象モデルの利用条件に同意済みであること


In [ ]:
# 必要なパッケージ
!pip install -q huggingface_hub kaggle

In [ ]:
import os, json, subprocess, time
from huggingface_hub import snapshot_download

MODEL_ID = "twolven/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-AWQ-MTP"
DATASET_ID = "yamahei21python/qwen38-27b-uncensored-hauhaucs-awq-mtp"
DATASET_SLUG = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
OUT_DIR = "/kaggle/working/qwen38-27b-uncensored-hauhaucs-awq-mtp"

# Kaggle Secrets から HF_TOKEN を取得
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if not HF_TOKEN:
    print("⚠ HF_TOKEN が未設定。 gated モデルはダウンロードできません。")
    HF_TOKEN = None
else:
    print("✅ HF_TOKEN を取得しました")

print(f"ダウンロード先: {OUT_DIR}")
print(f"モデル: {MODEL_ID}")


In [ ]:
# モデルダウンロード
# サイズ約 19GB。進捗が表示されるまで数分かかる場合あり。
start = time.time()
snapshot_download(
    repo_id=MODEL_ID,
    local_dir=OUT_DIR,
    local_dir_use_symlinks=False,
    token=HF_TOKEN,
    resume_download=True,
)
elapsed = time.time() - start
print(f"✅ ダウンロード完了: {elapsed/60:.1f}分")

# サイズ確認
!du -sh {OUT_DIR}
!ls -la {OUT_DIR}


In [ ]:
# Kaggle Dataset メタデータ作成
meta = {
    "title": "Qwen3.8-27B-Uncensored-HauhauCS-AWQ-MTP",
    "id": "yamahei21python/qwen38-27b-uncensored-hauhaucs-awq-mtp",
    "licenses": [{"name": "Apache 2.0"}],
}
with open(f"{OUT_DIR}/dataset-metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print(f"✅ dataset-metadata.json 作成: {OUT_DIR}/dataset-metadata.json")


In [ ]:
# Kaggle Dataset 作成
# 既存 Dataset がある場合は version コマンドに切り替える
import os
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

try:
    try:
        api.dataset_create_new(
            folder=OUT_DIR,
            convert_to_csv=False,
            dir_mode="zip",
        )
        print(f"✅ Dataset 作成完了: https://www.kaggle.com/datasets/{DATASET_ID}")
    except Exception as e:
        if "already exists" in str(e).lower() or "409" in str(e):
            print("既存 Dataset 検出。新バージョン作成中...")
            api.dataset_create_version(
                folder=OUT_DIR,
                version_notes="update from build_vllm_dataset",
                convert_to_csv=False,
                dir_mode="zip",
            )
            print(f"✅ Dataset 新バージョン作成完了: https://www.kaggle.com/datasets/{DATASET_ID}")
        else:
            raise
except Exception as e:
    print(f"⚠ 作成エラー: {e}")
    print("Kaggle 認証/kaggle.json、Dataset ID、権限を確認してください。")


---

## 完了後

Dataset URL: `https://www.kaggle.com/datasets/yamahei21python/qwen38-27b-uncensored-hauhaucs-awq-mtp`

この URL を `build_step2_vllm.py` の `DATASET_ID` に設定して Step2 notebook を生成する。
